In [1]:
import json
import pandas as pd
from pathlib import Path

In [2]:
dirs = "/workspace/worker/pj/Chrono/cortex/sphiex/records/attribution/test0/3"
file_path = Path(dirs)

In [19]:
evidence_units_list =[]
decision_rule_list =[]
invalidation_rules_list =[]
for json_file in file_path.rglob('*.json'):
    with json_file.open("r", encoding="utf-8") as file:
        history = json.load(file)
        evidence_units = history['trader_attribution']['experience_summary']['evidence_units']
        decision_rule = history['trader_attribution']['experience_summary']['decision_rule']
        invalidation_rules = history['trader_attribution']['experience_summary']['invalidation_rules']
        evidence_pd = pd.DataFrame(evidence_units)
        decision_pd = pd.DataFrame([decision_rule])
        invalidation_pd = pd.DataFrame(invalidation_rules)
        evidence_pd['date'] = history['trade_date']
        decision_pd['date'] = history['trade_date']
        invalidation_pd['date'] = history['trade_date']
        evidence_units_list.append(evidence_pd)
        decision_rule_list.append(decision_pd)
        invalidation_rules_list.append(invalidation_pd)
        

In [21]:
evidence_data = pd.concat(evidence_units_list,axis=0)
decision_data = pd.concat(decision_rule_list,axis=0)
invalidation_rules_data = pd.concat(invalidation_rules_list,axis=0)

In [26]:
evidence_data.head()

,evidence_id,feature_refs,feature_layer,interpretation,evidence_role,date
0,EV1,[main_flow_ratio],PREDICT,7/27 main_flow_ratio 由前日 -0.5257 大幅回升至 +0.4143...,DIRECTIONAL,2026-07-27
1,EV2,[price_ret_5d],PREDICT,7/27 price_ret_5d 由 7/24 的 0.0594 抬升至 0.5835，5...,DIRECTIONAL,2026-07-27
2,EV3,[comp_breadth_pos_20],PREDICT,comp_breadth_pos_20 由 -0.4246 修复至 +0.1489，上涨广度...,CONFIRMATION,2026-07-27
3,EV4,[lower_shadow_ratio],PREDICT,lower_shadow_ratio 由 -0.5025 收敛至 -0.3002，下影线风险...,CONFIRMATION,2026-07-27
4,EV5,[external_shock],TEXTUAL,TF-20260727-001 显示美暂停对伊空袭、伊朗暂停反击，油价大幅下跌；前期压制风险...,CONFIRMATION,2026-07-27


In [28]:
output_file = "analysis_result.xlsx"
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    evidence_data.to_excel(writer, sheet_name="证据单元", index=False)
    decision_data.to_excel(writer, sheet_name="决策规则", index=False)
    invalidation_rules_data.to_excel(writer, sheet_name="失效规则", index=False)
print(f"成功导出至 {output_file}")

成功导出至 analysis_result.xlsx


In [8]:
history['trader_attribution']['experience_summary'].keys()

dict_keys(['implied_direction', 'evidence_units', 'decision_rule', 'invalidation_rules', 'validation_status'])

In [10]:
history['trader_attribution']['experience_summary']['evidence_units']

[{'evidence_id': 'EV1',
  'feature_refs': ['main_flow_ratio'],
  'feature_layer': 'PREDICT',
  'interpretation': '07-27 main_flow_ratio 0.4143 转 07-28 -0.2840，叠加 07-24 -0.5257 的反复脉冲，资金呈现存量博弈而非趋势性撤离，不构成可靠做空动能。',
  'evidence_role': 'AUXILIARY'},
 {'evidence_id': 'EV2',
  'feature_refs': ['price_ret_5d'],
  'feature_layer': 'PREDICT',
  'interpretation': 'price_ret_5d 从 0.5835 降至 0.1271，但仍为正值；这是短线动量减速而非趋势反转，下方尚存惯性承接。',
  'evidence_role': 'CONFIRMATION'},
 {'evidence_id': 'EV3',
  'feature_refs': ['comp_breadth_pos_20'],
  'feature_layer': 'PREDICT',
  'interpretation': 'comp_breadth_pos_20 为 0.0903，虽边际回落但仍保持正值，市场宽度未确认系统性下行。',
  'evidence_role': 'CONFIRMATION'},
 {'evidence_id': 'EV4',
  'feature_refs': ['external_shock'],
  'feature_layer': 'TEXTUAL',
  'interpretation': 'PRE_T 已出现 TF-20260727-001 美伊暂停空袭、油价大幅下跌，07-28 又见 TF-20260728-003 国际油价大幅下跌；外部事件呈双向摇摆，不足以支撑持续单边避险。',
  'evidence_role': 'CONFIRMATION'},
 {'evidence_id': 'EV5',
  'feature_refs': ['domestic_policy'],
  'feature_layer': 'TE

In [11]:
history['trader_attribution']['experience_summary']['decision_rule']

{'required_evidence': ['EV2', 'EV4'],
 'supporting_evidence': ['EV1', 'EV3', 'EV5', 'EV6'],
 'conclusion': '因动量仅减速而未翻转、外部事件双向而非单向升级、政策托底与基本面韧性并存，单边做空不具备稳定优势；应保持 FLAT/防守，等待资金流、宽度和事件一致性出现明确方向后再介入。'}

In [12]:
history['trader_attribution']['experience_summary']['invalidation_rules']

[{'feature_refs': ['main_flow_ratio', 'comp_breadth_pos_20', 'price_ret_5d'],
  'condition': '若持有期内 main_flow_ratio 连续转正且 >0.3、comp_breadth_pos_20 回升至 0.3 以上、price_ret_5d 同步走强并保持在零轴上方，且政策与基本面未逆转。',
  'conclusion': 'FLAT 经验失效，转为谨慎偏多或恢复多头观察。'},
 {'feature_refs': ['main_flow_ratio',
   'price_ret_5d',
   'comp_breadth_pos_20',
   'external_shock'],
  'condition': '若 main_flow_ratio 持续 < -0.5、price_ret_5d 跌破零轴转负、comp_breadth_pos_20 转负，且外部冲击重新单向升级（如美伊冲突再升级、亚太系统性暴跌）。',
  'conclusion': 'FLAT 经验失效，转回防守性做空 / DOWN。'}]

In [13]:
history['trader_attribution']['experience_summary']['validation_status']

'UNVALIDATED_SINGLE_CASE'